# Notebook 08 — Bonus AI Workflow

**SPE Africa Geothermal Datathon 2026**

This notebook demonstrates an AI-assisted workflow that automates two key interpretation tasks:

1. **Automated Reservoir Description** — feeds `reservoir_properties.csv` into a large language model (LLaMA 3.3 70B via Groq API) and returns a professional written interpretation of each well's reservoir quality.
2. **Automated Risk Assessment** — feeds `lcoe_results.csv` and `system_design_summary.csv` into the model and returns a structured risk register with technical, economic, and operational risks plus mitigations.

Both outputs are saved as text files in `outputs/processed_data/`.

**Model:** `llama-3.3-70b-versatile` via Groq API  
**API key:** loaded from `.env` file — never hardcoded.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from groq import Groq

# Load environment variables from .env
load_dotenv()

# Directories
PROC_DIR = '../outputs/processed_data'
os.makedirs(PROC_DIR, exist_ok=True)

# Initialise Groq client
client = Groq(api_key=os.environ.get('GROQ_API_KEY'))
MODEL  = 'llama-3.3-70b-versatile'

print('Groq client initialised.')
print(f'Model: {MODEL}')

Groq client initialised.
Model: llama-3.3-70b-versatile


---
## Part 1 — Automated Reservoir Description

We load `reservoir_properties.csv`, format it as a structured prompt, and ask the model to produce a professional reservoir quality interpretation for each well.

In [2]:
# Load reservoir properties
df_res = pd.read_csv(os.path.join(PROC_DIR, 'reservoir_properties.csv'))
print('Reservoir properties loaded:')
print(df_res.set_index('Well').to_string())

Reservoir properties loaded:
        Rot_Top_m  Rot_Base_m  Gross_Thickness_m  Net_Pay_m   NTG  Log_Porosity_pct  Log_Perm_mD  Thermo_Porosity_pct  Thermo_Perm_mD_P50  Thermo_Perm_mD_P10  Thermo_Thickness_m_P50  Thermo_Temp_C_P50  Thermo_Flow_m3h_P50  Thermo_Power_MW_P50  Thermo_Power_MW_P10
Well                                                                                                                                                                                                                                                                   
BLT-01     1924.0      2123.0              199.0      138.2  0.69              14.0         40.4                 17.0                82.0               521.0                   130.0               77.0                  NaN                  5.1                 23.7
EVD-01     1788.0      2197.5              409.5      130.5  0.32              10.6         12.3                  9.0                 6.0                11.0                    76

In [3]:
def build_reservoir_prompt(df):
    """
    Convert the reservoir properties dataframe into a structured
    natural-language prompt for the LLM.
    """
    lines = []
    for _, row in df.iterrows():
        lines.append(
            f"Well: {row['Well']}\n"
            f"  Rotliegend interval: {row['Rot_Top_m']}m – {row['Rot_Base_m']}m\n"
            f"  Gross thickness: {row['Gross_Thickness_m']}m | Net pay: {row['Net_Pay_m']}m | NTG: {row['NTG']}\n"
            f"  Log porosity: {row['Log_Porosity_pct']}% | Log permeability: {row['Log_Perm_mD']} mD\n"
            f"  ThermoGIS porosity: {row['Thermo_Porosity_pct']}% | ThermoGIS perm P50: {row['Thermo_Perm_mD_P50']} mD | P10: {row['Thermo_Perm_mD_P10']} mD\n"
            f"  ThermoGIS thickness P50: {row['Thermo_Thickness_m_P50']}m | Temperature P50: {row['Thermo_Temp_C_P50']}°C\n"
            f"  ThermoGIS power P50: {row['Thermo_Power_MW_P50']} MW | P10: {row['Thermo_Power_MW_P10']} MW\n"
        )
    return '\n'.join(lines)


reservoir_data_text = build_reservoir_prompt(df_res)

reservoir_prompt = f"""You are a senior geothermal reservoir engineer preparing a technical assessment 
for a district heating project in the Netherlands (Utrecht region). The target formation is the 
Rotliegend sandstone.

Below are the reservoir characterisation results for four wells derived from wireline log analysis 
and the ThermoGIS probabilistic database:

{reservoir_data_text}

Write a professional reservoir quality interpretation covering ALL four wells. For each well:
1. Assess reservoir quality (porosity, permeability, net pay, NTG) and classify it as Excellent / Good / Fair / Poor.
2. Comment on the temperature and its suitability for direct geothermal heating.
3. State whether the well is viable as a production well at P50 and briefly explain why.
4. Note any key uncertainties or risks specific to that well.

End with a short paragraph summarising the overall resource quality and which wells should be 
prioritised for development.

Use clear, technical but accessible language suitable for a competition submission. 
Structure your response with a heading for each well."""

print('Reservoir prompt built.')
print(f'Prompt length: {len(reservoir_prompt)} characters')

Reservoir prompt built.
Prompt length: 2396 characters


In [4]:
print('Sending reservoir description request to Groq API...')

reservoir_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            'role': 'system',
            'content': (
                'You are a senior geothermal reservoir engineer with expertise in '
                'Dutch subsurface geology and the Rotliegend sandstone formation. '
                'Your reports are precise, well-structured, and technically rigorous.'
            )
        },
        {
            'role': 'user',
            'content': reservoir_prompt
        }
    ],
    max_tokens=2000,
    temperature=0.3   # low temperature = consistent, factual output
)

reservoir_description = reservoir_response.choices[0].message.content

print('Response received.\n')
print('=' * 70)
print('AI-GENERATED RESERVOIR DESCRIPTION')
print('=' * 70)
print(reservoir_description)

Sending reservoir description request to Groq API...
Response received.

AI-GENERATED RESERVOIR DESCRIPTION
## Well BLT-01
The reservoir quality of BLT-01 is classified as Good, with a log porosity of 14.0% and log permeability of 40.4 mD, indicating a relatively porous and permeable interval. The net pay of 138.2m and NTG of 0.69 further support this classification. The ThermoGIS porosity and permeability estimates are higher, with a P50 permeability of 82.0 mD, suggesting potential for even better reservoir performance. The temperature of 77.0°C is suitable for direct geothermal heating. At P50, the well is viable as a production well, with an estimated power output of 5.1 MW, due to its relatively high porosity and permeability. Key uncertainties for this well include the potential for variations in reservoir quality away from the wellbore and the accuracy of the ThermoGIS estimates.

## Well EVD-01
The reservoir quality of EVD-01 is classified as Poor, with a log porosity of 10.6% 

In [5]:
# Save reservoir description to file
reservoir_output_path = os.path.join(PROC_DIR, 'ai_reservoir_description.txt')

with open(reservoir_output_path, 'w') as f:
    f.write('SPE AFRICA GEOTHERMAL DATATHON 2026\n')
    f.write('AI-Assisted Reservoir Description\n')
    f.write(f'Model: {MODEL} via Groq API\n')
    f.write('=' * 70 + '\n\n')
    f.write(reservoir_description)

print(f'Reservoir description saved to: {reservoir_output_path}')

Reservoir description saved to: ../outputs/processed_data/ai_reservoir_description.txt


---
## Part 2 — Automated Risk Assessment

We load `lcoe_results.csv` and `system_design_summary.csv`, combine them into a structured prompt, and ask the model to produce a risk register covering technical, economic, and operational risks.

In [6]:
# Load LCOE results and system design summary
df_lcoe   = pd.read_csv(os.path.join(PROC_DIR, 'lcoe_results.csv'))
df_system = pd.read_csv(os.path.join(PROC_DIR, 'system_design_summary.csv'))

print('LCOE Results:')
print(df_lcoe.to_string(index=False))
print()
print('System Design Summary:')
print(df_system.to_string(index=False))

LCOE Results:
                              Metric   Value
                    Total CAPEX (M€)   30.34
                 Annual OPEX (k€/yr)  1433.6
    Annual energy delivered (MWh/yr) 66759.0
Annual electricity consumed (MWh/yr)  6938.0
                       Discount rate      6%
               Project lifetime (yr)      30
             Capital Recovery Factor  0.0726
              LCOE base case (€/MWh)    54.5
           LCOE P90 scenario (€/MWh)   125.1
           LCOE P50 scenario (€/MWh)    63.7
           LCOE P10 scenario (€/MWh)    63.2

System Design Summary:
                         Component                            Capacity                                         Role
  BLT-01 Production Well (doublet)                     105 m³/h @ 77°C               Primary geothermal heat source
  JUT-01 Production Well (doublet)                      55 m³/h @ 72°C             Secondary geothermal heat source
            Primary Heat Exchanger                           10.4 MWth Tra

In [7]:
def build_risk_prompt(df_lcoe, df_system):
    """
    Format LCOE and system design data into a structured risk assessment prompt.
    """
    lcoe_lines = '\n'.join(
        f"  {row['Metric']}: {row['Value']}" for _, row in df_lcoe.iterrows()
    )
    system_lines = '\n'.join(
        f"  {row['Component']} | Capacity: {row['Capacity']} | Role: {row['Role']}"
        for _, row in df_system.iterrows()
    )
    return lcoe_lines, system_lines


lcoe_text, system_text = build_risk_prompt(df_lcoe, df_system)

risk_prompt = f"""You are a senior energy systems risk analyst reviewing a geothermal district 
heating and cooling project in the Netherlands. The project targets a mixed urban district 
requiring 10 MWth heating and 5 MWth cooling from Rotliegend sandstone wells.

KEY PROJECT OUTCOMES:
  - Heating delivered: 10.39 MWth (target met)
  - Cooling delivered: 5.00 MWth (target met)
  - Primary well: BLT-01 (105 m³/h @ 77°C)
  - Secondary well: JUT-01 (55 m³/h @ 72°C)
  - EVD-01 and PKP-01 are non-viable at P50

ECONOMIC RESULTS:
{lcoe_text}

SYSTEM COMPONENTS:
{system_text}

Produce a structured project risk register with the following sections:

1. TECHNICAL RISKS — reservoir, wells, subsurface uncertainties
2. ECONOMIC RISKS — LCOE sensitivity, cost overruns, energy price changes
3. OPERATIONAL RISKS — system reliability, maintenance, grid dependency
4. REGULATORY / ENVIRONMENTAL RISKS — permitting, induced seismicity, groundwater

For each risk:
- Risk description (1-2 sentences)
- Likelihood: Low / Medium / High
- Impact: Low / Medium / High
- Mitigation measure (1-2 sentences)

Include at least 3 risks per section. End with an overall project risk rating 
(Low / Medium / High) with a brief justification.

Format clearly with section headings and numbered risks."""

print('Risk assessment prompt built.')
print(f'Prompt length: {len(risk_prompt)} characters')

Risk assessment prompt built.
Prompt length: 2351 characters


In [8]:
print('Sending risk assessment request to Groq API...')

risk_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            'role': 'system',
            'content': (
                'You are a senior energy systems risk analyst with expertise in '
                'geothermal projects, district heating, and Dutch energy regulations. '
                'Your risk registers are structured, actionable, and technically credible.'
            )
        },
        {
            'role': 'user',
            'content': risk_prompt
        }
    ],
    max_tokens=2500,
    temperature=0.3
)

risk_assessment = risk_response.choices[0].message.content

print('Response received.\n')
print('=' * 70)
print('AI-GENERATED RISK REGISTER')
print('=' * 70)
print(risk_assessment)

Sending risk assessment request to Groq API...
Response received.

AI-GENERATED RISK REGISTER
## 1. TECHNICAL RISKS — reservoir, wells, subsurface uncertainties

1. **Reservoir Depletion Risk**: The geothermal reservoir may deplete faster than expected, reducing the overall heat output and project lifespan. This could be due to inaccurate initial reservoir assessments or unforeseen changes in reservoir properties.
	* Likelihood: Medium
	* Impact: High
	* Mitigation measure: Regular monitoring of well performance and reservoir parameters, coupled with periodic reassessments of reservoir models to adjust extraction rates as necessary.
2. **Well Failure Risk**: The primary or secondary wells (BLT-01 or JUT-01) could fail due to mechanical issues, scaling, or other subsurface problems, leading to a significant reduction in heat supply. Well failure would necessitate costly repairs or even the drilling of new wells.
	* Likelihood: Medium
	* Impact: High
	* Mitigation measure: Implement a ri

In [9]:
# Save risk assessment to file
risk_output_path = os.path.join(PROC_DIR, 'ai_risk_assessment.txt')

with open(risk_output_path, 'w') as f:
    f.write('SPE AFRICA GEOTHERMAL DATATHON 2026\n')
    f.write('AI-Assisted Project Risk Register\n')
    f.write(f'Model: {MODEL} via Groq API\n')
    f.write('=' * 70 + '\n\n')
    f.write(risk_assessment)

print(f'Risk assessment saved to: {risk_output_path}')

Risk assessment saved to: ../outputs/processed_data/ai_risk_assessment.txt


---
## Summary

This notebook demonstrated an AI-assisted geothermal workflow using a large language model (LLaMA 3.3 70B via Groq API). Two automation tasks were completed:

| Task | Input | Output File |
|------|-------|-------------|
| Reservoir Description | `reservoir_properties.csv` | `ai_reservoir_description.txt` |
| Risk Register | `lcoe_results.csv` + `system_design_summary.csv` | `ai_risk_assessment.txt` |

### Why this workflow adds value
- Automates interpretation tasks that traditionally require hours of expert writing time
- Produces consistent, structured outputs directly from raw data files
- Fully reproducible — any team member can re-run with updated data and get a fresh interpretation
- Prompt engineering ensures domain-specific, technically grounded responses
- Low temperature setting (0.3) keeps outputs factual and consistent across runs

### Reproducibility note
Set `GROQ_API_KEY` in your `.env` file before running. The model and all parameters are fixed in code for full reproducibility.

In [10]:
print('=' * 70)
print('NOTEBOOK 08 COMPLETE')
print('=' * 70)
print()
print('Outputs saved:')
print(f'  1. {os.path.join(PROC_DIR, "ai_reservoir_description.txt")}')
print(f'  2. {os.path.join(PROC_DIR, "ai_risk_assessment.txt")}')
print()
print('AI workflow complete. All notebooks finished.')
print('Next: Build slide deck from notebook outputs.')

NOTEBOOK 08 COMPLETE

Outputs saved:
  1. ../outputs/processed_data/ai_reservoir_description.txt
  2. ../outputs/processed_data/ai_risk_assessment.txt

AI workflow complete. All notebooks finished.
Next: Build slide deck from notebook outputs.
